# 04 – Bee Haven: Gold-Tabelle (N4)

**Ziel:** Aus den bereinigten Silver-Daten **eine Gold-Tabelle je Stunde** bauen: Flugbewegungen, Stocktemperatur, Stockfeuchte, Gewicht und Außenwetter nebeneinander – die Grundlage für Dashboard, Analyse (N5) und maschinelles Lernen. Dazu die **Tageswerte** je Tag.

**Vorbild:** Kurs-Notebook 05 „Silver to Gold“ (alle Sensoren einer Stadt zu einer Tabelle zusammenführen). **Abweichungen** (Pflichtenheft PH-ITS-002, Karten 3.1 und 5.9):
- **Kurze Lücken werden erst hier in Gold aufgefüllt** – Silver bleibt die unveränderte Messung. Jede aufgefüllte Spalte bekommt eine Kennzeichen-Spalte `<spalte>_aufgefuellt` (True = geschätzt).
- **flow wird nie aufgefüllt** (Zählwerte lassen sich nicht aus Nachbarminuten ableiten).
- Nur Werte mit Qualität „ok…“ (Regeln R5–R7, R5a); auffällige Flugwerte (R9) zählen nicht mit.

**Einfach erklärt:** Silver ist das Kassenbuch (nur echte Messungen), Gold der Monatsbericht (übersichtlich zusammengefasst, Schätzungen gekennzeichnet).

**Läuft:** täglich in der Pipeline (Aktivität `n4_gold`); Parameter `STADT`.

> **Synapse-Version (Azure Synapse Analytics)** – automatisch erzeugt aus `../04_gold.ipynb` durch
> `Privat/03_notebooks_ITS-Data/generator/make_synapse_notebooks.py`. **Nicht von Hand ändern**, sondern das lokale
> Notebook ändern und den Generator erneut ausführen.
>
> Unterschiede zur lokalen Version (Vorbild: Kurs-Notebook 04 „weather pipeline synapse ready“):
> - Pfade als **ABFSS-URI (Azure Blob File System)** `abfss://<container>@<konto>.dfs.core.windows.net/...` statt `pathlib.Path`.
> - **Parameterzelle (Parameter Cell)** direkt darunter: eine Pipeline (Notebook-Aktivität) kann die Werte überschreiben.
> - **Kein Kontoschlüssel (Account Key) im Code:** Zugriff über die Identität des Synapse-Arbeitsbereichs (Workspace Identity)
>   bzw. einen verknüpften Dienst (Linked Service). Das Kurs-Notebook trägt den Schlüssel im Klartext ein – Sicherheitsrisiko.
> - Diagramme werden angezeigt; als PNG in den Lake nur bei `BILDER_SPEICHERN = True`.
> - Kosten: Ausführung nur nach Freigabe (Spark-Pool kostet je Minute).

In [ ]:
# --- Parameter (Parameters) – in Synapse als Parameterzelle markiert; Pipeline kann Werte überschreiben ---
STORAGE_ACCOUNT = "mabeehavenlakehouse"   # Speicherkonto (Storage Account, ADLS Gen2) des Lakehouse
CONTAINER = "lakehouse"                   # Container mit bronze/, silver/, gold/
STADT = "schwartau"                       # Standort = Unterordner in bronze/silver/gold und Teil der Dateinamen
LINKED_SERVICE = ""                       # leer = Identität des Arbeitsbereichs; sonst Name des verknüpften Dienstes (Linked Service)
BILDER_SPEICHERN = False                  # Diagramme zusätzlich als PNG nach gold/<stadt>/bilder/ schreiben?
LOKALER_TEST = ""                            # leer = Azure; lokaler Ordner (Spiegel des Containers) = Test ohne Azure

In [ ]:
# --- Hilfsfunktionen für Dateizugriffe: Azure Data Lake (abfss) oder lokaler Testordner ---
import os                                 # lokale Ordner (nur Testmodus)
import fnmatch                            # Dateinamen mit Platzhalter * vergleichen
import tempfile                           # Zwischenablage für PNG-Dateien

AZURE = not LOKALER_TEST                                                   # True = in Synapse gegen den Data Lake arbeiten
WURZEL = (f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"   # ABFSS-Wurzel des Containers (wie Kurs-Notebook 04)
          if AZURE else LOKALER_TEST.rstrip("/") + "/")                    # oder lokaler Testordner
SO = ({"linked_service": LINKED_SERVICE} if LINKED_SERVICE else {}) if AZURE else None   # storage_options für pandas
# Azure: SO nie None – sonst versucht pandas bei Parquet zuerst pyarrow direkt mit abfss:// und bricht ab
# ("Arrow compiled without Azure Blob File System support"); mit {} nimmt pandas den Synapse-Zugang über fsspec.
if AZURE:
    from notebookutils import mssparkutils                                 # Synapse-Werkzeuge: Dateien auflisten, Ordner anlegen, kopieren

def pfad(*teile):
    """Pfad unterhalb der Wurzel zusammensetzen, z. B. pfad("silver", "cleaned") -> abfss://.../silver/cleaned"""
    return WURZEL + "/".join(str(t).strip("/") for t in teile)

def dateien(ordner, muster="*"):
    """Sortierte Liste (voller Pfad, Größe in Byte) aller Dateien in EINEM Ordner, deren Name zum Muster passt."""
    if AZURE:
        try:
            eintraege = [(f.path, f.size) for f in mssparkutils.fs.ls(ordner) if f.isFile and fnmatch.fnmatch(f.name, muster)]
        except Exception:                                                  # Ordner existiert (noch) nicht
            eintraege = []
    else:
        eintraege = [(os.path.join(ordner, n), os.path.getsize(os.path.join(ordner, n)))
                     for n in (os.listdir(ordner) if os.path.isdir(ordner) else []) if fnmatch.fnmatch(n, muster)]
    return sorted(eintraege)                                               # sortiert nach Name -> letzte = neueste Version (Zeitstempel)

def neueste(ordner, muster):
    """Pfad der jüngsten Datei (Name enthält UTC-Zeitstempel) – Fehler, wenn keine vorhanden."""
    treffer = dateien(ordner, muster)
    if not treffer:
        raise FileNotFoundError(f"keine Datei {muster} in {ordner}")
    return treffer[-1][0]

def ordner_anlegen(ordner):
    """Zielordner anlegen, falls nicht vorhanden (in ADLS Gen2 unschädlich, wenn schon da)."""
    mssparkutils.fs.mkdirs(ordner) if AZURE else os.makedirs(ordner, exist_ok=True)

def archivieren(datei, neuer_name=None):
    """Datei in das _archiv ihres eigenen Ordners verschieben (Regel: jeder Ordner hat ein _archiv) – nichts wird gelöscht."""
    ordner, name = datei.rsplit("/", 1)
    ziel = f"{ordner}/_archiv/{neuer_name or name}"
    ordner_anlegen(f"{ordner}/_archiv")
    mssparkutils.fs.mv(datei, ziel) if AZURE else os.replace(datei, ziel)

def groesse_mb(datei):
    """Dateigröße in Megabyte (zur Kontrolle nach dem Speichern)."""
    ordner, name = datei.rsplit("/", 1)
    return next((g for p, g in dateien(ordner, name)), 0) / 1_000_000

def bild_speichern(fig, name, **optionen):
    """Diagramm optional als PNG in den Lake schreiben (Parameter BILDER_SPEICHERN)."""
    if not BILDER_SPEICHERN:
        return
    ziel_ordner = pfad("gold", STADT, "bilder")
    ordner_anlegen(ziel_ordner)
    lokal = os.path.join(tempfile.gettempdir(), name)                      # erst lokal auf dem Knoten speichern ...
    fig.savefig(lokal, **optionen)
    if AZURE:
        mssparkutils.fs.cp("file:" + lokal, ziel_ordner + "/" + name)       # ... dann in den Data Lake kopieren
    else:
        os.replace(lokal, os.path.join(ziel_ordner, name))

print("Wurzel:", WURZEL, "| Azure:", AZURE)

## 0. Bibliotheken und Pfade

In [ ]:
# --- Bibliotheken laden ---
import numpy as np                       # numpy: Rechnen mit Zahlenreihen
import pandas as pd                      # pandas: Tabellen zusammenführen und speichern

# --- Pfade festlegen ---
CLEANED = pfad("silver", "cleaned", STADT)                                         # Eingang: bereinigte Daten
GOLD = pfad("gold", STADT)                                                         # Ausgang: Gold-Tabellen
sensoren = ["flow", "humidity", "temperature", "weight"]                           # Liste der Sensoren

# --- Standortdaten aus der Konfiguration (Koordinaten, Zeitzone) ---
standorte = pd.read_csv(pfad("konfiguration", "standorte.csv"), index_col="stadt", storage_options=SO)
if STADT not in standorte.index:                          # unbekannte Stadt -> sofort abbrechen statt mit falschen Daten weiterrechnen
    raise ValueError(f"Standort '{STADT}' fehlt in konfiguration/standorte.csv – Zeile mit Koordinaten und Zeitzone ergänzen")
ORT = standorte.loc[STADT]                                # eine Zeile: name, breite, laenge, zeitzone, quelle
ZEITZONE = ORT["zeitzone"]                                # Ortszeit der Rohdaten (Deutschland: Europe/Berlin)

# --- Sensor-Regeln je Standort (Einheit der Rohdaten, gültiger Bereich) und bekannte Ausfallzeiträume ---
sensoren_cfg = pd.read_csv(pfad("konfiguration", "sensoren.csv"), storage_options=SO)          # eine Zeile je Stadt und Sensor
REGELN = sensoren_cfg[sensoren_cfg["stadt"] == STADT].set_index("sensor")              # Regeln dieser Stadt: einheit_roh, min, max
ausschluesse = pd.read_csv(pfad("konfiguration", "ausschluesse.csv"), storage_options=SO)      # bekannte Ausfallzeiträume
AUSSCHLUESSE = ausschluesse[ausschluesse["stadt"] == STADT].copy()                     # nur Zeiträume dieser Stadt
AUSSCHLUESSE[["von_utc", "bis_utc"]] = AUSSCHLUESSE[["von_utc", "bis_utc"]].apply(pd.to_datetime, utc=True)  # Text -> UTC-Zeit

print("Standort:", STADT, "·", ORT["name"], "· Zeitzone", ZEITZONE)

**Ergebnis:** Standort und Pfade gesetzt.

## 1. Neueste bereinigte Version einlesen

Je Sensor die jüngste Datei aus `silver/cleaned/<stadt>/<sensor>/` (Name enthält den UTC-Zeitstempel), dazu Wetter (stündlich) und Sonnenzeiten (täglich) aus N3.

In [ ]:
# --- Je Sensor die neueste Datei in cleaned finden und einlesen ---
daten = {}                                                              # leeres Dictionary: Sensor -> Tabelle
for s in sensoren:                                                      # für jeden Sensor ...
    versionen = dateien(f"{CLEANED}/{s}", f"{s}_{STADT}_cleaned__*.parquet")  # ... alle Versionen suchen (Name enthält Zeitstempel)
    datei = versionen[-1][0]                                            # ... letzte nach Namen = neueste Version
    daten[s] = pd.read_parquet(datei, storage_options=SO)                   # ... einlesen
    daten[s]["zeit_utc"] = pd.to_datetime(daten[s]["zeit_utc"], utc=True)  # ... CSV liefert Text -> wieder in UTC-Zeit umwandeln
    print(f"{s:12} {len(daten[s]):>10,} Zeilen  aus {datei.rsplit('/', 1)[-1]}  ({len(versionen)} Version(en) vorhanden)")  # ... Kontrolle

In [ ]:
# --- Neueste Wetter- und Sonnendaten aus N3 laden ---
neu = lambda ordner: neueste(f"{CLEANED}/{ordner}", f"{ordner}_{STADT}_cleaned__*.parquet")   # jüngste Version (Hilfszelle)
wetter = pd.read_parquet(neu("wetter"), storage_options=SO)                                   # stündlich
sonne = pd.read_parquet(neu("sonne"), storage_options=SO)   # täglich
wetter["zeit_utc"] = pd.to_datetime(wetter["zeit_utc"], utc=True)                                  # sicherstellen: UTC-Zeit
print(f"wetter {len(wetter):,} Stunden · sonne {len(sonne):,} Tage")

**Ergebnis:** Kontrollwerte wie in N2/N3 (flow 1.256.918 · humidity 1.761 · temperature 253.430 · weight 1.761 Zeilen; Wetter 21.145 Stunden).

## 2. Nur gültige Werte verwenden

Nur Zeilen mit Qualität „ok…“ (ungültige Werte aus R5, R5a, R6 und fehlende Werte fließen nicht ein). Bei flow zählen auffällige Werte (R9) nicht mit.

In [ ]:
# --- Nur Zeilen mit Qualität "ok..." behalten ---
gueltig = {}                                                            # leeres Dictionary für die gültigen Daten
for s in sensoren:                                                      # für jeden Sensor ...
    maske = daten[s]["qualitaet"].str.startswith("ok")                  # ... True, wenn die Qualität mit "ok" beginnt
    gueltig[s] = daten[s][maske]                                        # ... nur diese Zeilen übernehmen
    print(f"{s:12} {len(gueltig[s]):>10,} gültige Zeilen")              # ... Kontrolle

**Ergebnis:** gültige Zeilen je Sensor (weight 1.619 nach R5a).

## 3. Stundentabelle bauen

**Warum je Stunde?** Das Wetter liegt stündlich vor; Minuten (flow) und 5-Minuten-Werte (Temperatur) werden zur Stunde zusammengefasst, 12-Stunden-Werte (Feuchte, Gewicht) stehen in ihrer Messstunde. So passen alle Quellen auf **ein gemeinsames Zeitraster**.

**Eigene Erwartung:** ca. 21.100 Stunden (881 Tage × 24), Flugbewegungen bei Tag deutlich höher als nachts.

In [ ]:
# --- Gemeinsames Zeitraster: jede volle Stunde (UTC) vom ersten bis zum letzten Messwert ---
start = min(daten[s]["zeit_utc"].min() for s in sensoren).floor("h")    # früheste Messung, auf volle Stunde abgerundet
ende = max(daten[s]["zeit_utc"].max() for s in sensoren).floor("h")     # späteste Messung
gold = pd.DataFrame(index=pd.date_range(start, ende, freq="h", name="zeit_utc"))  # leere Tabelle, eine Zeile je Stunde

# --- flow: Flugbewegungen je Minute (Einflug + Ausflug als Betrag), auffällige Werte (R9) ausgeschlossen ---
fl = gueltig["flow"].set_index("zeit_utc")
ein = fl["flow_ein"].where(~fl["flow_ein_auffaellig"])                  # Einflug (positiv laut Datensatz-Beschreibung)
aus = fl["flow_aus"].abs().where(~fl["flow_aus_auffaellig"])            # Ausflug (negativ gemessen -> Betrag)
je_std = pd.DataFrame({"ein": ein, "aus": aus}).resample("h")           # nach Stunden gruppieren
gold["flug_ein_summe"] = je_std["ein"].sum(min_count=1)                 # Einflüge in der Stunde
gold["flug_aus_summe"] = je_std["aus"].sum(min_count=1)                 # Ausflüge in der Stunde
gold["flug_minuten"] = je_std["ein"].count()                            # wie viele Minuten gemessen wurden (max. 60)
gold["flug_bewegungen_pro_min"] = (gold["flug_ein_summe"] + gold["flug_aus_summe"]) / gold["flug_minuten"]  # Mittel je Minute

# --- Stocktemperatur: Stundenmittel der 5-Minuten-Werte ---
gold["stock_temperatur_c"] = gueltig["temperature"].set_index("zeit_utc")["temperature"].resample("h").mean()

# --- Feuchte und Gewicht: Messwert in seiner Messstunde (00:00 und 12:00 UTC) ---
gold["stock_feuchte_pct"] = gueltig["humidity"].set_index("zeit_utc")["humidity"].resample("h").mean()
gold["gewicht_kg"] = gueltig["weight"].set_index("zeit_utc")["weight"].resample("h").mean() / 1000   # Gramm -> kg

# --- Außenwetter der Stunde (N3) anhängen ---
spalten_wetter = ["aussentemperatur_c", "niederschlag_mm", "sonnenschein_min", "wind_kmh", "windboeen_kmh",
                  "luftdruck_hpa", "taupunkt_c", "sonnenstrahlung_kwh_m2", "wetterlage", "sonnenhoehe_grad", "tageszeit"]
gold = gold.join(wetter.set_index("zeit_utc")[spalten_wetter], how="left")
gold.insert(0, "stadt", STADT)                                          # Standort als Spalte (mehrere Städte in einer Tabelle möglich)
print(f"{len(gold):,} Stunden von {gold.index.min()} bis {gold.index.max()}")
gold.notna().mean().mul(100).round(1).to_frame("gefuellt_%").T         # Kontrolle: Anteil vorhandener Werte je Spalte

**Ergebnis:** 21.121 Stunden (01.01.2017 12:00 bis 31.05.2019 12:00 UTC) – wie erwartet ca. 21.100. Flug und Stocktemperatur zu 99,2 % gefüllt, Wetter zu 100 %; Feuchte (8,1 %) und Gewicht (7,7 %) nur in ihren Messstunden (Messtakt 12 Std.).

## 4. Kurze Lücken auffüllen (Gold-Regel, Entscheidung 30.09.2026)

**Regel:** Nur **kurze** Lücken bei **gleichmäßigen** Messgrößen werden zeitlich interpoliert (linear zwischen den Nachbarwerten); jede gefüllte Zelle ist in `<spalte>_aufgefuellt` gekennzeichnet. Längere Lücken bleiben leer.

| Spalte | höchstens aufgefüllte Lücke | Warum |
| :-- | :-- | :-- |
| `stock_temperatur_c` | 2 Stunden | Temperatur im Stock ändert sich langsam |
| `stock_feuchte_pct` | 11 Stunden (zwischen zwei 12-Stunden-Messungen) | fehlt eine Messung ganz, bleibt die Lücke |
| `gewicht_kg` | 11 Stunden | wie Feuchte; Zeitraum E5 (Stock nicht auf Waage) bleibt leer |
| flow-Spalten | nie | Zählwerte, nicht schätzbar |

**Vergleich Kurs-Notebook 04:** füllt schon in Silver (Temperatur bis 60 Werte = 5 Std., Gewicht bis 8 Werte = 4 Tage) – wir erst in Gold, mit Kennzeichen und kürzeren Grenzen.

**Eigene Erwartung:** Feuchte und Gewicht werden fast vollständig stündlich; Temperatur nur wenige zusätzliche Stunden.

In [ ]:
def kurze_luecken_fuellen(reihe, max_stunden):
    """Nur Lücken bis max_stunden aufeinanderfolgender fehlender Stunden zeitlich interpolieren.
    Gibt (gefüllte Reihe, Kennzeichen True = aufgefüllt) zurück; längere Lücken bleiben leer."""
    fehlt = reihe.isna()
    block = (fehlt != fehlt.shift()).cumsum()                            # Nummer je zusammenhängendem Block (fehlend / vorhanden)
    laenge = fehlt.groupby(block).transform("sum")                       # Länge des fehlenden Blocks, zu dem die Stunde gehört
    geschaetzt = reihe.interpolate(method="time", limit_area="inside")   # nur zwischen zwei echten Werten, nie am Rand
    erlaubt = fehlt & (laenge <= max_stunden) & geschaetzt.notna()       # kurze Lücke mit Nachbarwerten auf beiden Seiten
    return reihe.where(~erlaubt, geschaetzt), erlaubt

regeln = {"stock_temperatur_c": 2, "stock_feuchte_pct": 11, "gewicht_kg": 11}   # Spalte -> höchstens aufgefüllte Stunden
for spalte, max_std in regeln.items():
    gold[spalte], gold[f"{spalte}_aufgefuellt"] = kurze_luecken_fuellen(gold[spalte], max_std)
    print(f"{spalte:22} aufgefüllt: {gold[f'{spalte}_aufgefuellt'].sum():>6,} Stunden · weiter leer: {gold[spalte].isna().sum():>6,}")

**Ergebnis:** Temperatur 1 Stunde aufgefüllt (165 Stunden bleiben leer – längere Ausfälle); Feuchte 18.799 Stunden aufgefüllt (604 leer); Gewicht 17.765 Stunden aufgefüllt (1.737 leer, v. a. Zeitraum E5). Erwartung bestätigt: Feuchte und Gewicht sind danach fast durchgehend stündlich, lange Lücken bleiben leer.

## 5. Tageswerte

Je Kalendertag (UTC) Mittelwerte bzw. Summen – für Dashboard-Kacheln und Tagesvergleiche (vorher in der Analyse, jetzt hier in Gold).

In [ ]:
# --- Tageswerte je Sensor berechnen ---
tag = lambda t: t.set_index("zeit_utc").resample("D")                   # Hilfsfunktion: Daten nach Kalendertag (UTC) gruppieren
gewicht = tag(gueltig["weight"])["weight"].mean() / 1000                # Tagesmittel Gewicht, Gramm -> kg
fl_ok = gueltig["flow"].assign(                                         # flow ohne auffällige Werte (R9):
    flow_aus=gueltig["flow"]["flow_aus"].where(~gueltig["flow"]["flow_aus_auffaellig"]),  #   auffällige Ausflugwerte leeren
    flow_ein=gueltig["flow"]["flow_ein"].where(~gueltig["flow"]["flow_ein_auffaellig"]))  #   auffällige Einflugwerte leeren
ausflug = -tag(fl_ok)["flow_aus"].sum()                                 # Ausflüge je Tag (negativ gemessen -> positiv)
einflug = tag(fl_ok)["flow_ein"].sum()                                  # Einflüge je Tag
temp = tag(gueltig["temperature"])["temperature"].agg(["mean", "min", "max"])  # Temperatur: Mittel, Min, Max je Tag
feuchte = tag(gueltig["humidity"])["humidity"].mean()                   # Tagesmittel Luftfeuchte

tageswerte = pd.DataFrame({                                             # alles in EINE Tabelle zusammenführen:
    "gewicht_kg": gewicht,                                              #   Gewicht
    "ausflug": ausflug, "einflug": einflug,                             #   Flugaktivität
    "flugbewegungen": ausflug + einflug,                                #   Ein- + Ausflüge
    "temp_mittel": temp["mean"], "temp_min": temp["min"], "temp_max": temp["max"],  # Temperatur
    "feuchte_mittel": feuchte,                                          #   Luftfeuchte
}).round(2)                                                             # auf 2 Nachkommastellen runden
tageswerte.insert(0, "stadt", STADT)                                    # Standort als Spalte
print(len(tageswerte), "Tage von", tageswerte.index.min().date(), "bis", tageswerte.index.max().date())  # Kontrolle
tageswerte.describe().round(1)                                          # Überblick: Min, Max, Mittelwert je Kennzahl

**Ergebnis:** 881 Tage (Kontrollwert wie bisherige Gold-Vorschau).

## 6. Speichern nach `gold/<stadt>/`

Parquet (klein, Datentypen bleiben – Befund Z1) für Dashboard und Modelle, CSV zum Ansehen. Die alte Version wird vorher ins `_archiv` verschoben (nichts wird gelöscht).

In [ ]:
# --- Gold-Tabellen speichern ---
ordner_anlegen(GOLD)                                                    # Gold-Ordner anlegen, falls nicht vorhanden
stempel_gold = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d_%H%M%S")    # Zeitpunkt, bis zu dem die alte Version galt
for name in ["stunden", "tageswerte"]:
    for alt, _ in dateien(GOLD, f"{name}_{STADT}.*"):                   # alte Version vor dem Überschreiben ins _archiv
        stamm, endung = alt.rsplit("/", 1)[-1].rsplit(".", 1)
        archivieren(alt, f"{stamm}__bis_{stempel_gold}_UTC.{endung}")
gold.to_parquet(f"{GOLD}/stunden_{STADT}.parquet", storage_options=SO)  # Stundentabelle (Hauptergebnis)
gold.to_csv(f"{GOLD}/stunden_{STADT}.csv", storage_options=SO)          # zum Ansehen
tageswerte.to_parquet(f"{GOLD}/tageswerte_{STADT}.parquet", storage_options=SO)
tageswerte.to_csv(f"{GOLD}/tageswerte_{STADT}.csv", storage_options=SO)
for datei, groesse in dateien(GOLD, f"*_{STADT}.*"):                    # Kontrolle: Dateien und Größen
    print(datei.rsplit("/", 1)[-1], f"{groesse / 1000:.0f} KB")

**Ergebnis:** `stunden_schwartau.parquet` 972 KB (CSV 3.775 KB – Parquet ca. 26 %, vgl. Z1) · `tageswerte_schwartau.parquet` 47 KB.

## Zusammenfassung N4

| Tabelle | Inhalt | Datei |
| :-- | :-- | :-- |
| Stunden | Flug, Stocktemperatur, Stockfeuchte, Gewicht, Außenwetter, Tageszeit; Kennzeichen `_aufgefuellt` | `gold/<stadt>/stunden_<stadt>.parquet` |
| Tageswerte | Gewicht, Flug, Temperatur, Feuchte je Tag | `gold/<stadt>/tageswerte_<stadt>.parquet` |

Nächster Schritt: N5 Analyse (bei Bedarf) und Dashboard lesen aus Gold.